In [1]:
import polars as pl

pl.Config.set_tbl_rows(1000)    
pl.Config.set_tbl_cols(20)      
pl.Config.set_fmt_str_lengths(50)

polars.config.Config

In [2]:

keywords = [
    'сыр сливочный',
    'сыр кремовый',
    'сыр твердый',
    'сыр плавленый',
    'сыр творожный',
    'сыр полутвердый',
    'cream cheese',
    'cheese ',
    'сыр '
]

cheese_type_pattern = r'(?i)(' + '|'.join(keywords) + r')'
weight_pattern = r'(\d+)\s+г'
fat_percentage_pattern = r'(\d+)%'
raw_text_pattern = rf'{cheese_type_pattern}\s*([^\d\n]+)'

df = pl.scan_csv('./top_cheese_test.tsv', separator='\t').with_columns(pl.col('title').str.extract(raw_text_pattern, 2).alias('raw_text'))

raw_words_list = pl.col('raw_text').str.strip_chars().str.split(' ')
words_count = raw_words_list.list.len()
mid_idx = pl.max_horizontal(words_count // 2, 1)

normalized_df = (df.filter(pl.col('title').str.contains(cheese_type_pattern))
                 .with_columns(
                    pl.when(pl.col('title').str.contains(r'(?i)сливочный|cream cheese|кремовый'))
                    .then(pl.lit('сыр сливочный'))
                    .when(pl.col('title').str.contains(r'(?i)ванильный'))
                    .then(pl.lit('сыр ванильный'))
                    .when(pl.col('title').str.contains(r'(?i)творожный'))
                    .then(pl.lit('сыр творожный'))
                    .when(pl.col('title').str.contains(r'(?i)плавленый'))
                    .then(pl.lit('сыр плавленый'))
                    .otherwise(pl.lit('сыр обычный'))
                    .alias('cheese_type'),

                    pl.col('title').str.contains(r'(?i)возмещение').alias('is_canceled'),

                    pl.col('title').str.extract(weight_pattern, 1).alias('weight'),

                    pl.col('title').str.extract(fat_percentage_pattern, 1).alias('procenti'),

                    pl.when(pl.col('title').str.contains(r'(?i)сливочный|cream cheese|кремовый|творожный'))
                    .then(raw_words_list.list.slice(0, mid_idx).list.join(' '))
                    .otherwise(pl.col('raw_text'))
                    .alias('manufacturer'),
                    
                    pl.when(pl.col('title').str.contains(r'(?i)сливочный|cream cheese|кремовый|творожный'))
                    .then(raw_words_list.list.slice(mid_idx).list.join(' '))
                    .otherwise(None)
                    .alias('taste')
                 )
).drop('raw_text')

normalized_df.collect()

user_id,id,date_actual,title,sum,quantity,is_mm,check_id,cheese_type,is_canceled,weight,procenti,manufacturer,taste
i64,str,str,str,f64,f64,bool,str,str,bool,str,str,str,str
85633,"""0bad79b1-5372-4409-abb5-46b4a792e5a7""","""2023-04-07 12:45:26.000""","""сыр плавленый карат дружба 45% бзмж 400 г""",156.58,1.0,true,"""7b42ddcd-2ec3-4a21-b059-987e4644a3da""","""сыр плавленый""",false,"""400""","""45""","""карат дружба """,null
85633,"""1578dc2f-4ee7-4035-b87e-21d44a06c6ec""","""2023-01-26 09:48:48.000""","""сыр творожный hochland сливочный 60% 140 г""",119.99,1.0,false,"""4c78f077-a828-439a-90ac-04746211b232""","""сыр сливочный""",false,"""140""","""60""","""hochland""","""сливочный"""
85633,"""D5JvcI0BIXZkW_HSgS5I""","""2024-02-02 09:38:22.000""","""сыр творожный hochland сливочный 60% 140 г""",62.26,1.0,true,"""9289f2f7-b8fd-4dbb-93d9-25fef93c9c31""","""сыр сливочный""",false,"""140""","""60""","""hochland""","""сливочный"""
85633,"""512e8175-fb6c-4ff0-ade6-064cff7a4417""","""2023-07-15 11:20:14.000""","""возмещение за: сыр плавленый hochland сливочный 55…",242.26,1.0,false,"""3271d338-8247-4ab7-a76f-5aa4211ebd6f""","""сыр сливочный""",true,"""400""","""55""","""hochland""","""сливочный"""
85633,"""vVMoBosBCTUrFDcdfiRu""","""2023-10-05 10:35:01.000""","""возмещение за: сыр творожный hochland сливочный 60…",139.7,1.0,true,"""4aea2c39-8736-4bae-b02b-3672fdb8a424""","""сыр сливочный""",true,"""220""","""60""","""hochland""","""сливочный"""
85633,"""fWV8B4wBuO4rswBD3Yl_""","""2023-11-24 18:00:50.000""","""сыр творожный hochland сливочный 60% 140 г""",119.9,1.0,true,"""add49254-1017-471c-89b6-9dc372a0d82a""","""сыр сливочный""",false,"""140""","""60""","""hochland""","""сливочный"""
85633,"""1d9f8a38-429b-4174-8ea1-ba7bb7cfa185""","""2023-08-18 09:22:00.000""","""сыр творожный hochland с зеленью 60% 220 г""",139.99,1.0,true,"""a8dd8591-8d6c-4bea-800a-0426ad78e2d5""","""сыр творожный""",false,"""220""","""60""","""hochland""","""с зеленью"""
85633,"""8b6d012e-64d3-4147-b7f2-ed49c2fc9799""","""2023-07-19 16:02:57.000""","""возмещение за: сыр творожный hochland сливочный 60…",119.99,1.0,true,"""7fa1af37-a55c-4806-a8e6-abe960ecefdd""","""сыр сливочный""",true,"""140""","""60""","""hochland""","""сливочный"""
85633,"""-mqyGosBMQnv8mcBHzFD""","""2023-10-09 11:37:25.000""","""возмещение за: сыр творожный hochland сливочный 60…",139.39,1.0,true,"""b4036f21-eab0-4d54-9d6d-ecd60006bb4c""","""сыр сливочный""",true,"""220""","""60""","""hochland""","""сливочный"""
